# ST456 Colab 一键运行 Notebook（ZIP 上传版）

这个 notebook 对应当前项目的新版主线：

- E1-E5 是主实验
- retrieval 只作为 appendix / 附加实验
- 不需要 GitHub
- 只需要上传一个 ZIP 压缩包，然后点击 `Runtime -> Run all`


In [ ]:
# 参数区：运行前先改这里
ZIP_EXPECTED_HINT = 'ST456group.zip'
PROJECT_SUBDIR = 'codex-novel-continuation'

USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_WORKSPACE = '/content/drive/MyDrive/st456_runs'

# 当前默认先走快速验证模式：缩短训练和评测时间，先确认整条主线可用
QUICK_VALIDATION = True

# 全流程：跑 E1-E5 全部实验
RUN_FULL_MAINLINE = True

# retrieval 附加实验暂不运行
RUN_APPENDIX_RETRIEVAL = False

# 是否自动打包并下载 artifacts 结果
DOWNLOAD_RESULTS_ZIP = True

print('参数已加载（默认：Google Drive + 快速验证模式）。这个 notebook 运行时会要求你上传一个 ZIP 压缩包。')

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

# 减少模型加载时的 tqdm 进度条刷屏
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    """执行命令并实时显示输出，出错时抛出异常。"""
    print(f'\n>>> {cmd}', flush=True)
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True
    )
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(
            f'命令执行失败（exit code {process.returncode}）：{cmd}\n'
            f'错误信息已打印在上方，请向上滚动查看。'
        )

workspace_root = Path('/content')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    workspace_root = Path(GOOGLE_DRIVE_WORKSPACE)

workspace_root.mkdir(parents=True, exist_ok=True)
os.chdir(workspace_root)
print('当前工作目录:', Path.cwd())

from google.colab import files
print(f'请上传项目 ZIP 压缩包，例如: {ZIP_EXPECTED_HINT}')
uploaded = files.upload()
zip_candidates = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('没有检测到 ZIP 压缩包上传。')

zip_name = zip_candidates[0]
print('已上传 ZIP:', zip_name)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(workspace_root)

project_matches = sorted(workspace_root.rglob(PROJECT_SUBDIR))
project_matches = [path for path in project_matches if path.is_dir()]
if not project_matches:
    raise FileNotFoundError(f'解压之后没有找到 {PROJECT_SUBDIR} 目录。')

project_root = project_matches[0]
os.chdir(project_root)
print('项目目录:', Path.cwd())
run(f'{sys.executable} -m pip install -r requirements.txt')
print('环境准备完成。')

当前工作目录: /content
请上传项目 ZIP 压缩包，例如: ST456group.zip


Saving ST456group.zip to ST456group.zip
已上传 ZIP: ST456group.zip
项目目录: /content/ST456group/codex-novel-continuation

>>> /usr/bin/python3 -m pip install -r requirements.txt
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=731d6847cd2edbcdbf999eb91ff98175b48b183c3734734bc6702dd176f4c5c9
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
环境准备完成。


## 数据准备与 token 预算检查

这一步会：

- 下载 Sherlock Holmes 语料
- 构建数据集
- 保存 E1、E3、E5 的 token 统计结果


In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]

for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token 统计文件:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)



>>> /usr/bin/python3 scripts/download_gutenberg.py
Downloaded The Adventures of Sherlock Holmes -> data/raw/1661.txt
Downloaded The Memoirs of Sherlock Holmes -> data/raw/834.txt
Downloaded The Hound of the Baskervilles -> data/raw/2852.txt
Downloaded The Return of Sherlock Holmes -> data/raw/108.txt

>>> /usr/bin/python3 scripts/build_dataset.py --context-size 4 --min-chars 40

>>> /usr/bin/python3 scripts/inspect_token_stats.py --config configs/e1_distilgpt2_plain_full.yaml --output-path artifacts/eval/token_stats_e1.json
GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (1103 > 1024). Runn

## 主实验训练

E1 一定会跑。如果 `RUN_FULL_MAINLINE = True`，会继续跑 E2-E5。


In [ ]:
from novel_continuation.training import load_training_config

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
    ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
    ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ('E5W', 'configs/e5_distilgpt2_structured_aux_ranking_wide.yaml', 'artifacts/e5_aux_ranking_wide'),
]

for experiment_id, config_path, _output_dir in main_experiments:
    print(f'----- {experiment_id}: 开始训练 -----')
    run(f'{sys.executable} scripts/train_experiment.py --config {config_path} --seed 42')

print('主实验训练完成。')
for experiment_id, config_path, _output_dir in main_experiments:
    resolved = load_training_config(Path(config_path))
    print(f'- {experiment_id}: output_dir={resolved["output_dir"]}')


## 3-seed 样本生成与自动评估（人评可选）

这一步会为每个已经训练完成的主实验生成：

- `generated_samples_*_seed*.jsonl`
- `metrics_*_seed*.csv`
- `metrics_*_summary.csv`
- `human_eval_*.csv`（可选）


In [ ]:
import json
eval_output_dir = Path('artifacts/eval')
eval_output_dir.mkdir(parents=True, exist_ok=True)

for experiment_id, _config_path, model_dir in main_experiments:
    print(f'\n----- {experiment_id}: 3-seed 自动评估 -----')
    run(
        f'{sys.executable} scripts/run_eval_3seed.py ' \
        f'--experiment-id {experiment_id.lower()} ' \
        f'--model-dir {model_dir} ' \
        f'--output-dir {eval_output_dir}'
    )

print('\n3-seed 自动评估文件已生成。')
for path in sorted(eval_output_dir.glob('*')):
    print('-', path)

print('\n比较 E5 与 E5W 的 validation_main_loss：')
for experiment_id, _config_path, model_dir in main_experiments:
    if experiment_id not in {'E5', 'E5W'}:
        continue
    training_config = Path(model_dir) / 'training_config.json'
    if not training_config.exists():
        print(f'- {experiment_id}: 未找到 {training_config}')
        continue
    payload = json.loads(training_config.read_text(encoding='utf-8'))
    validation = payload.get('metadata', {}).get('validation', {})
    print(f'- {experiment_id}: validation_main_loss={validation.get("validation_main_loss")}, validation_perplexity={validation.get("validation_perplexity")}')


## Appendix：retrieval 附加实验

只有在 `RUN_APPENDIX_RETRIEVAL = True` 时才会执行这一块。


In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    # human-eval 导出保留为可选附加步骤，本轮默认不执行
    print('retrieval 附加实验已完成。')
else:
    print('已跳过 retrieval 附加实验。')


## 打包并下载结果

如果 `DOWNLOAD_RESULTS_ZIP = True`，会：

1. 清理中间 checkpoint（节省空间）
2. 只打包 eval 结果 + 训练配置（几 MB），不打包模型权重（几 GB）

In [7]:
import glob

if DOWNLOAD_RESULTS_ZIP:
    # 1. 清理中间 checkpoint 目录（节省 Colab 磁盘空间）
    for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
        if ckpt_dir.is_dir():
            shutil.rmtree(ckpt_dir)
            print(f'已清理: {ckpt_dir}')

    # 2. 收集需要下载的文件（eval 结果 + 每个实验的训练配置）
    download_dir = Path('artifacts/download_package')
    download_dir.mkdir(parents=True, exist_ok=True)

    # 复制 eval 目录（metrics、生成样本、人评 CSV、token stats）
    eval_src = Path('artifacts/eval')
    if eval_src.exists():
        shutil.copytree(eval_src, download_dir / 'eval', dirs_exist_ok=True)

    # 复制每个实验的 training_config.json（记录超参数，写报告用）
    for config_file in Path('artifacts').glob('*/training_config.json'):
        dest = download_dir / config_file.parent.name / config_file.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_file, dest)

    # 3. 打包并下载
    archive_path = shutil.make_archive(
        str(Path('artifacts') / 'colab_results'), 'zip',
        root_dir=str(download_dir)
    )
    size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f'压缩包已生成: {archive_path} ({size_mb:.1f} MB)')

    # 清理临时目录
    shutil.rmtree(download_dir)

    from google.colab import files
    files.download(archive_path)
else:
    print('已跳过结果压缩包下载。')

已清理: artifacts/e1_plain_full/checkpoint-661
已清理: artifacts/e2_structured_full/checkpoint-661
已清理: artifacts/e3_long_context/checkpoint-661
已清理: artifacts/e4_lora/checkpoint-661
已清理: artifacts/e5_aux_ranking/checkpoint-661
压缩包已生成: /content/ST456group/codex-novel-continuation/artifacts/colab_results.zip (0.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>